# IBKR 阶梯费用对齐与分钟模型结构诊断

用户确认 Tiered、10 万美元。冻结当前 NN/控制器，17 个原生账户、0 次新拟合。每股佣金、逐单最低/最高费、月度阶梯、卖出监管费及场所费用分别列账。0/2/5bp 仅为未知成交滑点压力；当前价目表投影到旧历史，不能作为历史账单或前向业绩。

In [1]:
from pathlib import Path
import sys, os, json, pickle
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"AGENTS.md").exists() and (p/"src").is_dir())
sys.path.insert(0,str(ROOT))
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
import qlib
from qlib.data import D
from qlib.workflow import R
from IPython.display import display,Markdown
from src.workflow.ibkr_minute_diagnosis import prepare_ibkr_minute_diagnosis,run_ibkr_minute_diagnosis
CARD=ROOT/"research/ibkr_tiered_minute_diagnosis_20261013"
PLAN=json.loads((CARD/"plan.json").read_text())
qlib.init(provider_uri=str(ROOT/PLAN["adjusted_provider"]),region="us",kernels=1)
assert len(D.calendar(start_time=PLAN["signal_start"],end_time=PLAN["as_of"]+" 16:00:00",freq="15min"))>20000
PLAN["user_confirmed"]

'IBKR Pro Tiered, USD 100000'

## 数据和费用基准

从 NAS 的固定 Qlib Recorder 读市场、目标、模型和训练标签；raw NONE 价格也通过 Qlib D 读取。风险动作继续调用现役函数，每个费用/滑点账户独立重放现金、持仓和风险时钟。整股变化以原价换算，权益 NAV 保留父研究复权记账平面；不隐含实际分红现金、税务或 broker 持股资格。

In [2]:
CACHE=Path("/tmp/soxl-ibkr-minute-20261013/prepared.pkl")
if (CARD/"summary.json").exists():
    PREPARED=None
elif CACHE.exists():
    with CACHE.open("rb") as h: PREPARED=pickle.load(h)
else:
    PREPARED=prepare_ibkr_minute_diagnosis(root=ROOT,card=CARD,plan=PLAN)
    with CACHE.open("wb") as h: pickle.dump(PREPARED,h)
if PREPARED is not None: display({"sessions":len(PREPARED["market"]),"price_provider":str(PREPARED["provider_uri"])})

{'sessions': 944,
 'price_provider': '/Users/rziiiii/dev/workspace/data/qlib/ibkr_tiered_minute_diagnosis_20261013'}

In [3]:
SUMMARY=json.loads((CARD/"summary.json").read_text()) if PREPARED is None else run_ibkr_minute_diagnosis(root=ROOT,card=CARD,plan=PLAN,prepared=PREPARED)
assert SUMMARY["new_model_fits"]==0 and SUMMARY["native_cases"]==17
assert SUMMARY["current_release_changed"] is False
SUMMARY["direct_minute_head_audit"]

{'observations': 946,
 'minute_numerical_channels': 12,
 'maximum_tail_logit_change_when_minute_channels_neutralized': 0.0,
 'maximum_alpha_logit_change_when_minute_channels_neutralized': 0.0,
 'mean_absolute_allocation_logit_change': 0.28348731994628906,
 'minute_features_directly_enter_risk_or_alpha_heads': False,
 'explanation': 'tail_encoder(common_features) supplies both heads; minute channels only enter allocation_head',
 'minute_information_clock': 'previous completed session, next daily open; not a live minute-frequency action model'}

In [4]:
with R.uri_context(SUMMARY["recorder_uri"]):
    REC=R.get_recorder(experiment_id=SUMMARY["experiment_id"],recorder_id=SUMMARY["recorder_id"])
    COMPARISON=REC.load_object("analysis/comparison")
    CALIBRATION=REC.load_object("analysis/calibration")
display(COMPARISON[["family","slippage_bps","return_pct","max_drawdown_pct","relative_wealth","notional_weighted_broker_and_external_bps","broker_and_external_fees_usd","orders","recovery_actions"]])
display(CALIBRATION)

,family,slippage_bps,return_pct,max_drawdown_pct,relative_wealth,notional_weighted_broker_and_external_bps,broker_and_external_fees_usd,orders,recovery_actions
0,frozen_current,0,233.080895,-26.668568,0.667682,1.343774,1346.310071,1145,0
1,base,0,283.464861,-26.501056,0.768680,1.215511,1472.930802,1110,0
2,availability_control,0,78.004275,-29.171115,0.356821,1.441600,1245.248105,1077,0
3,core_path,0,328.925685,-25.488036,0.859809,1.226486,1649.542582,1225,0
4,soxx,0,398.861540,-42.738804,1.000000,0.190960,1.904383,1,0
5,frozen_current,2,229.141547,-26.733271,0.659812,1.348938,1339.897722,1144,0
6,base,2,278.345821,-26.674945,0.758449,1.219051,1465.757557,1110,0
7,availability_control,2,105.091369,-29.066399,0.411135,1.358312,1208.907096,1072,0
8,core_path,2,270.530746,-25.555249,0.742782,1.264164,1559.691150,1210,0
9,soxx,2,398.841595,-42.742502,1.000000,0.190960,1.904383,1,0


,family,signal_year,target,observations,auc,brier,event_frequency,mean_probability,role
0,base,2023,tail_event_5d,250,0.192828,0.043244,0.028000,0.145236,reused-history calibration diagnosis; no model...
1,base,2023,relative_alpha_5d,250,0.521700,0.248563,0.528000,0.518154,reused-history calibration diagnosis; no model...
2,base,2024,tail_event_5d,252,0.614318,0.100489,0.111111,0.157247,reused-history calibration diagnosis; no model...
3,base,2024,relative_alpha_5d,252,0.418887,0.251931,0.559524,0.522775,reused-history calibration diagnosis; no model...
4,base,2025,tail_event_5d,250,0.637615,0.109455,0.128000,0.136387,reused-history calibration diagnosis; no model...
5,base,2025,relative_alpha_5d,250,0.602476,0.236713,0.616000,0.542639,reused-history calibration diagnosis; no model...
6,base,2026,tail_event_5d,188,0.584737,0.160079,0.202128,0.206095,reused-history calibration diagnosis; no model...
7,base,2026,relative_alpha_5d,188,0.417483,0.254008,0.585106,0.504125,reused-history calibration diagnosis; no model...
8,availability_control,2023,tail_event_5d,250,0.194591,0.043221,0.028000,0.145203,reused-history calibration diagnosis; no model...
9,availability_control,2023,relative_alpha_5d,250,0.519774,0.248615,0.528000,0.518201,reused-history calibration diagnosis; no model...


In [5]:
display(Markdown((CARD/"conclusion.md").read_text()))

# IBKR 阶梯费率与分钟模型诊断

使用用户确认的 IBKR Pro Tiered、10 万美元本金；逐资产整股下单。佣金用真实 NONE 价格的股数逐单计算，不能用复权记账单位直接乘每股费率。
采用查询日现行价目表投影到相同历史窗口，并假设 SMART 订单按 Nasdaq 流动性移除费收费、每单一次成交、其他账户成交量为零。0/2/5bp 是单独列出的滑点现金成本情景，不是 IBKR 佣金。TAF 当前假期费率为零，CAT/清算/卖出 SEC 与佣金附加费分别列账；保留浮点美元，不声明账单美分取整或税务资格。

| 完整策略 | Tiered + 2bp 滑点：累计净收益 | 最大回撤 | 佣金及附加费等效 bp |
|---|---:|---:|---:|
| frozen_current | 229.14% | -26.73% | 1.349 |
| base | 278.35% | -26.67% | 1.219 |
| availability_control | 105.09% | -29.07% | 1.358 |
| core_path | 270.53% | -25.56% | 1.264 |
| soxx | 398.84% | -42.74% | 0.191 |

全部滑点情景、两种仓位/置信度互换诊断见 backtest_benchmark_comparison.csv；互换项不能晋升。风险状态依据各账户自己的真实费用、现金和净值重算；Native Qlib 逐单费用、现金、数量、下一开盘 NAV 与独立账本核对。
结构证据：将核心分钟数值通道中性化，风险和 alpha logits 的变化严格为零，而仓位 logits 改变。原共享 tail_encoder 仅读取公共日频特征；新增分钟数据只进入 allocation_head。概率年度 AUC/Brier 见 probability_calibration.csv，反映同一已复用历史，不能选新模型。
限制：权益 NAV 沿用父研究的复权记账平面；整股表示的是每次交易金额对应的 raw 股数，未实现实际分红现金/税务/跨除权持股及真实 09:31/10:31 成交复核。结果是统一的当前费用投影，不是 IBKR 历史实盘账单或独立前向超额。
晋升：not_promoted_fee_alignment_and_structure_diagnosis；模型未重训、现役模拟盘未切换，独立合格前向为 0。
后续：先让分钟信息进入专门的风险/超额预测路径，并把信息时点、预测期限与实际动作统一；训练成本也要使用本次逐单口径。在冻结的有限实验中与原结构、可用性对照比较，禁止根据本窗口改阈值或换候选。
Recorder：19b476e0dea34abead078565d8f739c6；数据、订单、持仓和大报告保留在 NAS。

## 成本与结构结论

实际成交金额加权的佣金及附加费为约 1.22–1.44bp，不能直接用 10/25bp 当作 IBKR 佣金。整股数量及新的净费用同时改变风险路径，因此本表与旧连续仓位压力情景的差异不能只解释为静态扣费减少。核心分钟模型 0bp 滑点时为 328.93%/−25.49%，同结构无分钟为 283.46%/−26.50%；2bp 时为 270.53%/−25.56% 对 278.35%/−26.67%；5bp 时为 105.17%/−29.23% 对 261.52%/−26.94%。各场景均未跑赢 SOXX，分钟输入不是毫无作用，但没有获得稳定净超额。

两个冻结置信度互换对照的收益、回撤和订单数完全一致，所有账户恢复动作均为 0。中性化 12 项分钟数值对风险和 alpha logits 的改变为 0，对仓位 logits 的平均绝对改变为 0.28349。这个接法是上一交易日分钟摘要驱动日频分配，不是真正的日内动作价值模型。

原训练仍使用 10bp 费用和 0.006 换手正则，且不重放完整 Stage1 风控；alpha 的复用诊断 AUC 在 2024/2026 约 0.419/0.418。成本口径、信息进入的头、预测期限与实际动作都需要对齐，不能把一次低成本回测的改善当作新模型晋升。

下一轮已具体写入 next_experiment.json，状态 prepared_not_run：只在现有 10:31 时点比较状态输入与状态加分钟输入的净执行价值，固定两个 Ridge10 对照、三个年度截止点，最多 6 次拟合、12 个账户；强制减仓和持仓约束保留。先验证同状态 execute/HOLD 的真实净差及标签成熟时点，再考虑扩展下午时点；新前向数据要记录报价接收时点、bid/ask 和实际费用。


# IBKR 阶梯费率与分钟模型诊断

使用用户确认的 IBKR Pro Tiered、10 万美元本金；逐资产整股下单。佣金用真实 NONE 价格的股数逐单计算，不能用复权记账单位直接乘每股费率。
采用查询日现行价目表投影到相同历史窗口，并假设 SMART 订单按 Nasdaq 流动性移除费收费、每单一次成交、其他账户成交量为零。0/2/5bp 是单独列出的滑点现金成本情景，不是 IBKR 佣金。TAF 当前假期费率为零，CAT/清算/卖出 SEC 与佣金附加费分别列账；保留浮点美元，不声明账单美分取整或税务资格。

| 完整策略 | Tiered + 2bp 滑点：累计净收益 | 最大回撤 | 佣金及附加费等效 bp |
|---|---:|---:|---:|
| frozen_current | 229.14% | -26.73% | 1.349 |
| base | 278.35% | -26.67% | 1.219 |
| availability_control | 105.09% | -29.07% | 1.358 |
| core_path | 270.53% | -25.56% | 1.264 |
| soxx | 398.84% | -42.74% | 0.191 |

全部滑点情景、两种仓位/置信度互换诊断见 backtest_benchmark_comparison.csv；互换项不能晋升。风险状态依据各账户自己的真实费用、现金和净值重算；Native Qlib 逐单费用、现金、数量、下一开盘 NAV 与独立账本核对。
结构证据：将核心分钟数值通道中性化，风险和 alpha logits 的变化严格为零，而仓位 logits 改变。原共享 tail_encoder 仅读取公共日频特征；新增分钟数据只进入 allocation_head。概率年度 AUC/Brier 见 probability_calibration.csv，反映同一已复用历史，不能选新模型。
限制：权益 NAV 沿用父研究的复权记账平面；整股表示的是每次交易金额对应的 raw 股数，未实现实际分红现金/税务/跨除权持股及真实 09:31/10:31 成交复核。结果是统一的当前费用投影，不是 IBKR 历史实盘账单或独立前向超额。
晋升：not_promoted_fee_alignment_and_structure_diagnosis；模型未重训、现役模拟盘未切换，独立合格前向为 0。
后续：先让分钟信息进入专门的风险/超额预测路径，并把信息时点、预测期限与实际动作统一；训练成本也要使用本次逐单口径。在冻结的有限实验中与原结构、可用性对照比较，禁止根据本窗口改阈值或换候选。
Recorder：19b476e0dea34abead078565d8f739c6；数据、订单、持仓和大报告保留在 NAS。

## 成本与结构结论

实际成交金额加权的佣金及附加费为约 1.22–1.44bp，不能直接用 10/25bp 当作 IBKR 佣金。整股数量及新的净费用同时改变风险路径，因此本表与旧连续仓位压力情景的差异不能只解释为静态扣费减少。核心分钟模型 0bp 滑点时为 328.93%/−25.49%，同结构无分钟为 283.46%/−26.50%；2bp 时为 270.53%/−25.56% 对 278.35%/−26.67%；5bp 时为 105.17%/−29.23% 对 261.52%/−26.94%。各场景均未跑赢 SOXX，分钟输入不是毫无作用，但没有获得稳定净超额。

两个冻结置信度互换对照的收益、回撤和订单数完全一致，所有账户恢复动作均为 0。中性化 12 项分钟数值对风险和 alpha logits 的改变为 0，对仓位 logits 的平均绝对改变为 0.28349。这个接法是上一交易日分钟摘要驱动日频分配，不是真正的日内动作价值模型。

原训练仍使用 10bp 费用和 0.006 换手正则，且不重放完整 Stage1 风控；alpha 的复用诊断 AUC 在 2024/2026 约 0.419/0.418。成本口径、信息进入的头、预测期限与实际动作都需要对齐，不能把一次低成本回测的改善当作新模型晋升。

下一轮已具体写入 next_experiment.json，状态 prepared_not_run：只在现有 10:31 时点比较状态输入与状态加分钟输入的净执行价值，固定两个 Ridge10 对照、三个年度截止点，最多 6 次拟合、12 个账户；强制减仓和持仓约束保留。先验证同状态 execute/HOLD 的真实净差及标签成熟时点，再考虑扩展下午时点；新前向数据要记录报价接收时点、bid/ask 和实际费用。
